In [12]:
import json
import os
import pandas as pd
import re
import shutil
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from functions import *
from eval_functions import *

In [13]:
main_directory = 'model_output'
models = find_folders_with_output(main_directory)

save = False

eval_path = f"evaluate/batch1"
os.makedirs(eval_path, exist_ok=True)

### Model Output to nice JSON and Failure 

In [40]:
def process_files(model, save=save):
    input_dir = f"model_output/{model}/output/"
    output_dir = f"model_output/{model}/formatted/"
    failure_dir = f"model_output/{model}/failed/"
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(failure_dir, exist_ok=True)
    files = os.listdir(input_dir)
    len_files = len(files)
    for filename in files:
        if filename.endswith('.json'):
            input_file_path = os.path.join(input_dir, filename)
            output_file_path = os.path.join(output_dir, filename)
            failure_file_path = os.path.join(failure_dir, filename)
            try:
                file = read_json(input_file_path)
                #print(f"Processing file: {filename}")
                if save:
                    shutil.copy(input_file_path, output_file_path)
                    #save_json_to_file(file, output_file_path)
            except Exception as e:
                #print(f"Error processing file {filename}: {e}")
                if save:
                    shutil.copy(input_file_path, failure_file_path)
    return len_files

In [53]:
for model in models:
    print(model)
    # 1 - Preprocess Files
    num_out_files = process_files(model, save=save)
    # 2 - Evaluate Files
    p2_label_path = "chia_label/p2"
    ready_path = f"model_output/{model}/ready"
    failed_model_path = f"model_output/{model}/failed_inner"
    p2_model_formatted_path = f"model_output/{model}/formatted"
    log_filename =  "missing_words_logfile.txt"
    for path in [ready_path, failed_model_path, p2_model_formatted_path]:
        os.makedirs(path, exist_ok=True)
    
    label_files = {extract_nct_number(f): os.path.join(p2_label_path, f) for f in os.listdir(p2_label_path) if f.endswith('.json')}
    model_files = {extract_nct_number(f): os.path.join(p2_model_formatted_path, f) for f in os.listdir(p2_model_formatted_path) if f.endswith('.json')}
    
    common_ncts = set(label_files.keys()).intersection(model_files.keys())
    labels = []
    predictions = []
    success_data = []
    for nct in common_ncts:
        try:
            label_data = read_json(label_files[nct])
            model_data = read_json(model_files[nct])
            label_structure = extract_logical_structure(label_data)
            model_structure = extract_logical_structure(model_data)
            # Versuche Wörter zu zählen
            label_raw_texts = extract_raw_texts(label_data)
            model_raw_texts = extract_raw_texts(model_data)
            label_words = extract_words(label_raw_texts)
            model_words = extract_words(model_raw_texts)
            missing_words = label_words - model_words
            missing_words_pct = len(missing_words) / len(label_words) * 100 if label_words else 0
    
            with open(log_filename, 'a', encoding='utf-8') as outfile:
                outfile.write(f"{nct}; [{model}] Missing Words: {round(missing_words_pct, 2)} %   ;  Is Subset:  {label_words.issubset(model_words)}\n")
                outfile.write("Label Text \n")
                outfile.write(f"{label_words} \n")
                outfile.write("\nMissing Words \n")
                outfile.write(f"{missing_words}\n")
                outfile.write("\nModel Text \n")
                outfile.write(f"{model_words}\n")
                outfile.write("\n\n")
            
            success_data.append({
                'NCT': nct,
                'label_AND': label_structure.get('AND', 0),
                'label_OR': label_structure.get('OR', 0),
                'label_NOT': label_structure.get('NOT', 0),
                'label_DEPTH': label_structure.get('depth', 0),
                'model_AND': model_structure.get('AND', 0),
                'model_OR': model_structure.get('OR', 0),
                'model_NOT': model_structure.get('NOT', 0),
                'model_DEPTH': model_structure.get('depth', 0),
                'diff_AND': 1 if model_structure.get('AND', 0) > label_structure.get('AND', 0) else -1 if model_structure.get('AND', 0) < label_structure.get('AND', 0) else 0,
                'diff_OR': 1 if model_structure.get('OR', 0) > label_structure.get('OR', 0) else -1 if model_structure.get('OR', 0) < label_structure.get('OR', 0) else 0,
                'diff_NOT': 1 if model_structure.get('NOT', 0) > label_structure.get('NOT', 0) else -1 if model_structure.get('NOT', 0) < label_structure.get('NOT', 0) else 0,
                'diff_DEPTH': 1 if model_structure.get('depth', 0) > label_structure.get('depth', 0) else -1 if model_structure.get('depth', 0) < label_structure.get('depth', 0) else 0,
                'num_out_files': num_out_files,
            })
            labels.append(label_structure)
            predictions.append(model_structure)
            save and shutil.copy(model_files[nct], os.path.join(ready_path, os.path.basename(model_files[nct])))
        except Exception as e:
            #print(f"Error processing NCT {nct}: {e}")
            save and shutil.copy(model_files[nct], os.path.join(failed_model_path, os.path.basename(model_files[nct])))
    
    df_success = pd.DataFrame(success_data).set_index('NCT')
    df_success.to_csv(eval_path+f'/{model}_eval.csv')
    true_values = df_success[['label_AND', 'label_OR', 'label_NOT', 'label_DEPTH']].values
    predicted_values = df_success[['model_AND', 'model_OR', 'model_NOT', 'model_DEPTH']].values
    
    metrics = {}
    for i, metric in enumerate(['AND', 'OR', 'NOT', 'DEPTH']):
        y_true = true_values[:, i]
        y_pred = predicted_values[:, i]
    
        diffs = y_pred - y_true
        pct_greater = (diffs > 0).sum() / len(diffs) * 100
        pct_less = (diffs < 0).sum() / len(diffs) * 100
        pct_equal = (diffs == 0).sum() / len(diffs) * 100
    
        metrics[metric] = {
            'accuracy': round(accuracy_score(y_true, y_pred), 3),
            'precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'recall': round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'f1_score': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'pct_greater': round(pct_greater, 2),
            'pct_less': round(pct_less, 2),
            'pct_equal': round(pct_equal, 2)
            # 'confusion_matrix': confusion_matrix(y_true, y_pred)
        }
        metrics_df = pd.DataFrame(metrics).T
        num_nct_files = len(df_success)
        metrics_df['num_nct_files'] = num_nct_files
        metrics_df['model_name'] = model
        metrics_df['num_files'] = num_out_files
        # Save Metrics to CSV
        metrics_df.to_csv(os.path.join(eval_path, f'{model}_metrics_summary.csv'))    

Llama-3-70B-Instruct_5_shot
Llama-3-8B-Instruct_4_shot


In [ ]:
# NCT02935855_inc

In [ ]:
df_success = pd.DataFrame(success_data).set_index('NCT')
df_success.to_csv(eval_path+f'/{model}_eval.csv')

df_success

In [ ]:
true_values = df_success[['label_AND', 'label_OR', 'label_NOT', 'label_DEPTH']].values
predicted_values = df_success[['model_AND', 'model_OR', 'model_NOT', 'model_DEPTH']].values

metrics = {}
for i, metric in enumerate(['AND', 'OR', 'NOT', 'DEPTH']):
    y_true = true_values[:, i]
    y_pred = predicted_values[:, i]

    diffs = y_pred - y_true
    pct_greater = (diffs > 0).sum() / len(diffs) * 100
    pct_less = (diffs < 0).sum() / len(diffs) * 100
    pct_equal = (diffs == 0).sum() / len(diffs) * 100


    metrics[metric] = {
        'accuracy': round(accuracy_score(y_true, y_pred), 3),
        'precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'recall': round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'f1_score': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'pct_greater': round(pct_greater, 2),
        'pct_less': round(pct_less, 2),
        'pct_equal': round(pct_equal, 2)
       # 'confusion_matrix': confusion_matrix(y_true, y_pred)
    }
    metrics_df = pd.DataFrame(metrics).T  
    num_nct_files = len(df_success)
    metrics_df['num_out_files'] = num_out_files
    metrics_df['num_nct_files'] = num_nct_files
    metrics_df['model_name'] = model
    # Save Metrics to CSV
    metrics_df.to_csv(os.path.join(eval_path, f'{model}_metrics_summary.csv'))

print(f"{len(df_success)} Daten mit {model}")
for metric, values in metrics.items():
    print(f"Metrics for {metric}:")
    print(f"  Accuracy: {values['accuracy']}")
    print(f"  Precision: {values['precision']}")
    print(f"  Recall: {values['recall']}")
    print(f"  F1 Score: {values['f1_score']}")

    print(f"  % Greater: {values['pct_greater']}")
    print(f"  % Less: {values['pct_less']}")
    print(f"  % Equal: {values['pct_equal']}")
    print()
    #print(f"  Confusion Matrix:\n{values['confusion_matrix']}\n")

In [ ]:
metrics_df

In [ ]:
matching_rows = df_success[df_success['label_AND'] == df_success['model_AND']][['label_AND', 'model_AND']]
matching_rows